# Clase Práctica de pandas: Ejercicios GUIADOS

## Parte 1: EDA básico (sin gráficas)
En esta sección realizaremos las primeras exploraciones de los datos para entender su estructura.

In [ ]:
import pandas as pd

# Cargar los datasets
df_house = pd.read_csv('/mnt/data/house_price.csv')
df_divisa = pd.read_csv('/mnt/data/divisa.csv')

### Ejercicio 1: Primer vistazo
- Usa `head()` para ver las primeras filas de cada DataFrame.
- Aplica `info()` para conocer tipos y valores no nulos.
- Ejecuta `describe()` para estadísticos descriptivos.
- Calcula `isnull().sum()` para identificar valores faltantes.

In [ ]:
# Primer vistazo a 'house_price'
df_house.head()

df_house.info()

df_house.describe()

df_house.isnull().sum()

In [ ]:
# Primer vistazo a 'divisa'
df_divisa.head()

df_divisa.info()

df_divisa.describe()

df_divisa.isnull().sum()

## Parte 2: Renombrar columnas
Aprenderemos a cambiar nombres de columnas de forma programática.

### Ejercicio 2a: Usando un bucle `for`
- Renombra todas las columnas de `df_house` a minúsculas y reemplaza espacios por guiones bajos.
- ¿Por qué usar un bucle? Permite estandarizar patrones de nombre sin especificar cada columna.

In [ ]:
for col in df_house.columns:
    new_name = col.lower().replace(' ', '_')
    df_house.rename(columns={col: new_name}, inplace=True)
df_house.columns

### Ejercicio 2b: Usando una función y `map()`
- Define una función `clean_col(name)` que transforme el nombre.
- Aplica `df_divisa.columns = df_divisa.columns.map(clean_col)`.
- Ventaja: reutilización en otros DataFrames.

In [ ]:
def clean_col(name):
    return name.strip().lower().replace('.', '').replace(' ', '_')

df_divisa.columns = df_divisa.columns.map(clean_col)
df_divisa.columns

## Parte 3: Operaciones `groupby` usando funciones
Veremos cómo encapsular operaciones de agregación en funciones.

### Ejercicio 3a: Precio medio por número de habitaciones
- Crea la función `avg_price_by_bedrooms(df)` que devuelva la media de `price` agrupada por `bedrooms`.
- ¿Por qué usar funciones? Facilitan reutilización y tests.

In [ ]:
def avg_price_by_bedrooms(df):
    return df.groupby('bedrooms')['price'].mean()

avg_price_by_bedrooms(df_house)

### Ejercicio 3b: Tipo de divisa promedio por mes
- Crea la función `avg_rate_by_month(df)` que:
  1. Convierte la columna `date` a datetime.
  2. Extrae el mes como período (`dt.to_period('M')`).
  3. Agrupa por ese período y calcula la media de `rate`.
- Explica por qué es útil agrupar por periodos de tiempo.

In [ ]:
def avg_rate_by_month(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df['month'] = df['date'].dt.to_period('M')
    return df.groupby('month')['rate'].mean()

avg_rate_by_month(df_divisa)

## Parte 4: Gestión de valores faltantes con `fillna`
Exploraremos distintos métodos para imputar datos faltantes.

### Ejercicio 4a: Imputación con la mediana
- Usa `df_house['column'].fillna(df_house['column'].median(), inplace=True)`.
- ¿Por qué la mediana? Es robusta a valores extremos.

In [ ]:
# Ejemplo para la columna 'price' de df_house
df_house['price'].fillna(df_house['price'].median(), inplace=True)
df_house['price'].isnull().sum()

### Ejercicio 4b: Imputación hacia adelante y hacia atrás
- Aplica `df_divisa['rate'].fillna(method='ffill')` y luego `bfill`.
- Discute cuándo usar cada método.

In [ ]:
# Forward fill
df_divisa['rate_ffill'] = df_divisa['rate'].fillna(method='ffill')
# Backward fill
df_divisa['rate_bfill'] = df_divisa['rate'].fillna(method='bfill')
df_divisa[['rate', 'rate_ffill', 'rate_bfill']].head(10)

## Parte 5: Gestión de duplicados
Identificaremos y removeremos filas duplicadas.

### Ejercicio 5a: Detección de duplicados
- Usa `df.duplicated()` para marcar duplicados en `df_house`.
- ¿Qué criterio consideras para definir duplicado?

In [ ]:
# Marcar duplicados en todo el DataFrame
df_house['is_dup'] = df_house.duplicated()
df_house[['is_dup']].value_counts()

### Ejercicio 5b: Eliminación de duplicados
- Aplica `df_house.drop_duplicates(keep='first', inplace=True)` y luego con `keep='last'`.
- Explica la diferencia entre ambas.

In [ ]:
# Eliminar duplicados, conservar primero
df_house_clean1 = df_house.drop_duplicates(keep='first')
# Eliminar duplicados, conservar último
df_house_clean2 = df_house.drop_duplicates(keep='last')
len(df_house), len(df_house_clean1), len(df_house_clean2)

### Ejercicio 5c: Duplicados en `df_divisa`
- Detecta y elimina duplicados basados en combinación de `date` y `currency`.
- ¿Por qué podríamos querer considerar solo ciertas columnas?